# 03 — Model Training & Evaluation

Train and compare:
1. Logistic Regression (baseline)
2. XGBoost with Optuna hyperparameter tuning
3. LightGBM

All models use `TimeSeriesSplit` cross-validation to prevent data leakage.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna
import mlflow
import warnings
from pathlib import Path
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from src.features.actuarial_features import ActuarialFeatureBuilder
from src.models.evaluate import (
    compute_business_metrics, plot_lift_curve, plot_roc_curve, plot_calibration,
    full_evaluation_report,
)

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_PATH = Path("../data/churn_dataset.parquet")

In [ ]:
df = pd.read_parquet(DATA_PATH)
X = df.drop(columns=["churn_label", "policy_id"])
y = df["churn_label"]

builder = ActuarialFeatureBuilder()
X_features = builder.fit_transform(X)

drop_cols = ["lob", "channel"]
X_model = X_features.drop(columns=[c for c in drop_cols if c in X_features.columns])

split_idx = int(len(X_model) * 0.80)
X_train, X_test = X_model.iloc[:split_idx], X_model.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {len(X_train):,} | Test: {len(X_test):,} | Churn rate: {y.mean():.1%}")

## 1. Baseline — Logistic Regression

In [ ]:
baseline = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
baseline.fit(X_train, y_train)

y_prob_base = baseline.predict_proba(X_test)[:, 1]
report_base = full_evaluation_report(y_test, y_prob_base)
print("Baseline (Logistic Regression):")
for k, v in report_base.items():
    print(f"  {k}: {v}")

## 2. XGBoost + Optuna Tuning

In [ ]:
cv = TimeSeriesSplit(n_splits=5)

def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 5),
        "random_state": 42,
        "eval_metric": "auc",
        "verbosity": 0,
    }
    model = XGBClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective_xgb, n_trials=30, show_progress_bar=True)

print(f"Best CV AUC: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

In [ ]:
best_params = study.best_params
best_params.update({"random_state": 42, "eval_metric": "auc", "verbosity": 0})
xgb_model = XGBClassifier(**best_params)
xgb_model.fit(X_train, y_train)

calibrated_xgb = CalibratedClassifierCV(xgb_model, method="sigmoid", cv="prefit")
calibrated_xgb.fit(X_train, y_train)

y_prob_xgb = calibrated_xgb.predict_proba(X_test)[:, 1]
report_xgb = full_evaluation_report(y_test, y_prob_xgb)
print("XGBoost (calibrated):")
for k, v in report_xgb.items():
    print(f"  {k}: {v}")

## 3. LightGBM

In [ ]:
lgbm_model = LGBMClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    num_leaves=31,
    class_weight="balanced",
    random_state=42,
    verbosity=-1,
)
lgbm_model.fit(X_train, y_train)

y_prob_lgbm = lgbm_model.predict_proba(X_test)[:, 1]
report_lgbm = full_evaluation_report(y_test, y_prob_lgbm)
print("LightGBM:")
for k, v in report_lgbm.items():
    print(f"  {k}: {v}")

## Model Comparison

In [ ]:
comparison = pd.DataFrame({
    "Logistic Regression": {"AUC-ROC": report_base["auc_roc"], "Avg Precision": report_base["avg_precision"], "Brier": report_base["brier_score"]},
    "XGBoost": {"AUC-ROC": report_xgb["auc_roc"], "Avg Precision": report_xgb["avg_precision"], "Brier": report_xgb["brier_score"]},
    "LightGBM": {"AUC-ROC": report_lgbm["auc_roc"], "Avg Precision": report_lgbm["avg_precision"], "Brier": report_lgbm["brier_score"]},
}).T
comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for y_prob, name, ax in [
    (y_prob_base, "Logistic Regression", axes[0]),
    (y_prob_xgb, "XGBoost", axes[1]),
    (y_prob_lgbm, "LightGBM", axes[2]),
]:
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, linewidth=2, label=f"AUC = {auc:.3f}")
    ax.plot([0, 1], [0, 1], "--", color="gray")
    ax.set_title(name)
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")
    ax.legend()

plt.suptitle("ROC Curves — Model Comparison", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig = plot_lift_curve(y_test, y_prob_xgb)
plt.show()